In [2]:
from datetime import datetime, timedelta
import pandas as pd
import os
import re
import ee
import math
import geemap
import calendar

In [3]:
ee.Authenticate()
ee.Initialize()

In [3]:
input_folder = r"C:\Users\方慈弘\Desktop\YellowRiver\Restart\temp_input"
output_folder = r"C:\Users\方慈弘\Desktop\YellowRiver\Restart\temp_output"

In [4]:
result = pd.DataFrame(columns = ['id','Time','dissolved_oxygen','turbidity','phosphorus','nitrogen','chlorophyll',
                                 'lon_wgs84','lat_wgs84','image_id','formatted_date','time_diff',
                                 'Blue','Green','Red','Nir','Swir1','Swir2',
                                 'temp','prec','wind','solar','evaporation','lai_h','lai_l',
                                 'temp_mon','prec_mon','wind_mon','solar_mon','evaporation_mon','lai_h_mon','lai_l_mon'])

In [5]:
# MatchUp 2 fmask
def fmask(image):
     # see https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2
     # Bit 0 - Fill
     # Bit 1 - Dilated Cloud
     # Bit 2 - Cirrus
     # Bit 3 - Cloud
     # Bit 4 - Cloud Shadow
    cloudsBitMask = (1 << 5)
    snowBitMask = (1 << 4)
    cloudshadow = (1 << 3)
    qa = image.select("BQA")
    qaMask = qa.bitwiseAnd(cloudsBitMask).eq(0)\
            .And(qa.bitwiseAnd(snowBitMask).eq(0))\
            .And(qa.bitwiseAnd(cloudshadow).eq(0)).rename('QA')

    # Apply the scaling factors to the appropriate bands.
    opticalBands = image.select(['Blue', 'Green', 'Red', 'Swir1', 'Nir', 'Swir2']).multiply(0.0000275).add(-0.2)

    # Replace the original bands with the scaled ones and apply the masks..addBands(opticalBands, None, True)
    return image.addBands(qaMask).addBands(opticalBands, None, True).updateMask(qaMask)

In [6]:
def get_images(s_t, e_d, poi):
    geometry = poi.buffer(45)
    
    # 仅使用 Landsat 7 波段
    bn8 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B6', 'QA_PIXEL', 'SR_B5', 'SR_B7']
    bn7 = ['SR_B1', 'SR_B1', 'SR_B2', 'SR_B3', 'SR_B5', 'QA_PIXEL', 'SR_B4', 'SR_B7']
    bn9 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B6', 'QA_PIXEL', 'SR_B5', 'SR_B7']
    bns = ['uBlue', 'Blue', 'Green', 'Red', 'Swir1', 'BQA', 'Nir', 'Swir2']

    # 选择 Landsat 7 Collection 2 Level 2
    ls7 = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2").select(bn7, bns)
    ls8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2").select(bn8, bns)
    ls9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2").select(bn9, bns)
    merged = ls7.merge(ls8).merge(ls9)

    # 过滤日期和区域
    imgcol = merged.filterDate(s_t, e_d).filterBounds(poi)
    
    # 过滤云量小于 80% 的影像
    imgcol_cloud = imgcol.filterMetadata('CLOUD_COVER', 'less_than', 80)

    # 影像预处理（掩膜 + 裁剪）
    def clip_and_mask(image):
        return fmask(image).clip(geometry)

    processed_images = imgcol_cloud.map(clip_and_mask)

    # 统计影像数量
    count = processed_images.size().getInfo()
    print(f"Landsat 7 影像数量: {count}")
    
    return processed_images

In [7]:
def find_closest_data(target_date):
    # 将目标日期转换为 datetime 格式
    target_date = datetime.strptime(target_date, '%Y-%m-%d %H:%M:%S')
    
    # 计算时间差（以小时为单位）
    df['time_diff'] = abs((df['Time'] - target_date).dt.total_seconds()) / 3600
    closest_index = df['time_diff'].idxmin()

    # 读取该行的所需数据
    closest_data = df.loc[closest_index, ['id', 'Time', 'dissolved_oxygen', 'turbidity', 'phosphorus', 'nitrogen', 'chlorophyll']]
    
    # 计算时间差（以小时为单位）
    closest_date = df.loc[closest_index, 'Time']
    time_difference = abs((closest_date - target_date).total_seconds()) / 3600

    # 检查时间差是否大于3天（72小时）
    if time_difference > 72:
        marked = True
    else:
        marked = False

    return closest_data, marked, time_difference

In [8]:
# 遍历文件夹中的每个文件
for file_name in os.listdir(input_folder):

    # .csv文件需要更换读取方式
    file_path = os.path.join(input_folder, file_name)
    df = pd.read_csv(file_path, encoding='GBK')


     # 将"Time"列转换为 datetime 格式
    df['Time'] = pd.to_datetime(df['Time'])

    #观测的时间范围和空间范围
    start_date = df['Time'].min()
    end_date = df['Time'].max()
    print(start_date, end_date)
    x = df['lon_wgs84'].iloc[0]
    y = df['lat_wgs84'].iloc[0]
    poi = ee.Geometry.Point([x, y])

    # 获取时间和空间范围内的所有影像
    processed_images = get_images(start_date, end_date, poi)

    # 将 ImageCollection 转换为 Python 列表 
    # 在 Google Earth Engine 中，通常使用 map 方法来对影像集合中的每个影像应用一个函数。若想要显式地遍历每个影像并对其进行处理，需要将其转换为 python list
    image_list = processed_images.toList(processed_images.size())

    # 遍历每个影像
    for i in range(image_list.size().getInfo()):
        image = ee.Image(image_list.get(i))
    
        # 输出image_id以检查
        image_id = image.get('system:index').getInfo()
        print(image_id, end=' ')
    
        geometry = poi.buffer(45)
        pixel_value = image.reduceRegion(reducer = ee.Reducer.median(), geometry = geometry, scale = 10, maxPixels = 1e9).getInfo()
        if pixel_value.get('Blue') is None:
            print('X')
            continue
            
        # reflectance
        keys_of_interest = ['Blue', 'Green', 'Red', 'Nir', 'Swir1', 'Swir2']
        values_list = [pixel_value[key] for key in keys_of_interest]
    
        # timestap of image
        timestamp = image.get('system:time_start').getInfo()
        date_ls = ee.Date(timestamp)
        formatted_date = date_ls.format('yyyy-MM-dd HH:mm:ss').getInfo()

        # 匹配日期最接近的 in-situ data。若与日期最接近的 in-situ data 的时间差 > 3天，则跳过
        cl_data, mark ,time_diff= find_closest_data(formatted_date)
        if mark:
            print('0')
            continue

        # era5_land daily
        era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')\
                 .filterBounds(geometry)\
                 .filter(ee.Filter.date(date_ls, date_ls.advance(1, 'day'))).first().clip(geometry).reduceRegion(reducer=ee.Reducer.mean(),geometry=geometry,scale=30).getInfo()
        temp = era5.get('temperature_2m')
        prec = era5.get('total_precipitation_sum')
        wind_u = era5.get('u_component_of_wind_10m')
        wind_v = era5.get('v_component_of_wind_10m')
        wind = math.sqrt(wind_u**2+wind_v**2)
        solar = era5.get('surface_net_solar_radiation_sum')
        evaporation = era5.get('evaporation_from_open_water_surfaces_excluding_oceans_sum')
        lai_h = era5.get('leaf_area_index_high_vegetation')
        lai_l = era5.get('leaf_area_index_low_vegetation')
    
        #era5_land monthly
        era_year = date_ls.get("year").getInfo()
        era_mon = date_ls.get("month").getInfo()
        last_day_of_month = calendar.monthrange(era_year, era_mon)[1]
        start_date = f'{era_year}-{era_mon}-01'
        end_date = f'{era_year}-{era_mon}-{last_day_of_month}'
        era5_mon = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')\
                 .filterBounds(geometry)\
                 .filter(ee.Filter.date(start_date, end_date)).first().clip(geometry).reduceRegion(reducer=ee.Reducer.mean(),geometry=geometry,scale=30).getInfo()
        temp_mon = era5_mon.get('temperature_2m')
        prec_mon = era5_mon.get('total_precipitation_sum')
        wind_u_mon = era5_mon.get('u_component_of_wind_10m')
        wind_v_mon = era5_mon.get('v_component_of_wind_10m')
        wind_mon = math.sqrt(wind_u_mon**2+wind_v_mon**2)
        solar_mon = era5_mon.get('surface_net_solar_radiation_sum')
        evaporation_mon = era5_mon.get('evaporation_from_open_water_surfaces_excluding_oceans_sum')
        lai_h_mon = era5_mon.get('leaf_area_index_high_vegetation')
        lai_l_mon = era5_mon.get('leaf_area_index_low_vegetation')
    
        # output
        row_list = cl_data.tolist()+[x,y,image_id,formatted_date,time_diff]+values_list+[temp,prec,wind,solar,evaporation,lai_h,lai_l,temp_mon,prec_mon,wind_mon,solar_mon,evaporation_mon,lai_h_mon,lai_l_mon]
        result.loc[len(result)] = pd.Series(row_list, index=result.columns)
        print('1')

    output_path = os.path.join(output_folder, file_name)
    result.to_csv(output_path, index=False)
    print(f"{file_name} 处理完成!")

    #清空 result
    result = pd.DataFrame(columns = ['id','Time','dissolved_oxygen','turbidity','phosphorus','nitrogen','chlorophyll',
                                     'lon_wgs84','lat_wgs84','image_id','formatted_date','time_diff','Blue','Green','Red','Nir','Swir1','Swir2',
                                     'temp','prec','wind','solar','evaporation','lai_h','lai_l',
                                     'temp_mon','prec_mon','wind_mon','solar_mon','evaporation_mon','lai_h_mon','lai_l_mon'])

print("所有文件处理完成!")

2021-03-10 20:00:00 2022-12-31 20:00:00
Landsat 7 影像数量: 96
1_1_LE07_125036_20210321 X
1_1_LE07_125036_20210406 X
1_1_LE07_125036_20210422 X
1_1_LE07_125036_20210508 X
1_1_LE07_125036_20210524 1
1_1_LE07_125036_20210727 X
1_1_LE07_125036_20210913 1
1_1_LE07_125036_20210929 1
1_1_LE07_125036_20211116 1
1_1_LE07_125036_20211202 1
1_1_LE07_125036_20211218 1
1_1_LE07_125036_20220103 1
1_1_LE07_125036_20220119 1
1_1_LE07_125036_20220204 X
1_1_LE07_125036_20220220 1
1_1_LE07_125036_20220308 1
1_1_LE07_125036_20220510 X
1_1_LE07_125036_20220815 X
1_1_LE07_125036_20220901 0
1_1_LE07_125036_20220918 X
1_1_LE07_125036_20221217 0
1_1_LE07_126036_20210312 X
1_1_LE07_126036_20210328 1
1_1_LE07_126036_20210413 X
1_1_LE07_126036_20210429 X
1_1_LE07_126036_20210531 X
1_1_LE07_126036_20210803 X
1_1_LE07_126036_20210920 1
1_1_LE07_126036_20211022 X
1_1_LE07_126036_20211107 X
1_1_LE07_126036_20211123 1
1_1_LE07_126036_20211209 1
1_1_LE07_126036_20220110 X
1_1_LE07_126036_20220227 1
1_1_LE07_126036_2022031